# Synthetic transaction data generation

In this notebook I'll create reproducible synthetic customer, account, merchant and transaction data using numpy

## Objectives

- Generate behavioural profiles for synthetic customers
- Create related account and merchant tables
- Generate transactions using vectorised NumPy operations
- Model customer specific amounts, hours, categories and international activity
- Inject known behavioural anomalies
- Preserve anomaly ground truth for later evaluation
- Prepare scalable dataset sizes for future performance testing

Data quality errors and behavioural anomalies are different, as data quality errors are invalid or corrupted records while behavioural anomalies are valid records that are unusual for a customer

In [1]:
from pathlib import Path 
import numpy as np 
import pandas as pd

In [2]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

synthetic_directory = (
    project_root / "data" / "synthetic" / "sample"
)

synthetic_directory.mkdir(parents=True, exist_ok=True)

In [3]:
SEED = 42

rng = np.random.default_rng(SEED)

In [4]:
dataset_sizes = {"debug": 100,
                 "development": 1_000,
                 "integration": 100_000,
                 "benchmark": 1_000_000
                 }

customer_counts = {
    "debug": 20,
    "development": 100,
    "integration": 5_000,
    "benchmark": 50_000,
}

DATASET_TIER = "development"

N_TRANSACTIONS = dataset_sizes[DATASET_TIER]

N_CUSTOMERS = customer_counts[DATASET_TIER]

In [5]:
# I will use arrays as catalogues

countries = np.array(
    [
        "Spain",
        "France",
        "Germany",
        "Italy",
        "Portugal",
        "Ireland",
        "United Kingdom",
        "United States",
    ]
)

currencies = np.array(
    [
        "EUR",
        "EUR",
        "EUR",
        "EUR",
        "EUR",
        "EUR",
        "GBP",
        "USD",
    ]
)

segments = np.array(
    [
        "Student",
        "Standard",
        "Premium",
    ]
)

categories = np.array(
    [
        "Groceries",
        "Restaurants",
        "Books",
        "Electronics",
        "Sports",
        "Travel",
        "Home",
        "Entertainment",
        "Fashion",
        "Health",
    ]
)

n_countries = len(countries)
n_categories = len(categories)

In [6]:
# Ids, country and segment

customer_ids = np.array([f"C{i:05d}" for i in range(1, N_CUSTOMERS + 1)])

customer_ids[:5]

array(['C00001', 'C00002', 'C00003', 'C00004', 'C00005'], dtype='<U6')

In [7]:
customer_country_idx = rng.integers(0, n_countries, size = N_CUSTOMERS)

In [8]:
customer_country_idx[:5]

array([0, 6, 5, 3, 3])

In [9]:
countries[customer_country_idx[:5]]

array(['Spain', 'United Kingdom', 'Ireland', 'Italy', 'Italy'],
      dtype='<U14')

In [10]:
customer_segment_idx = rng.choice(
    len(segments),
    size = N_CUSTOMERS,
    p = [0.20, 0.55, 0.25]
)

In [11]:
segments[customer_segment_idx[:10]]

array(['Student', 'Student', 'Premium', 'Standard', 'Standard', 'Premium',
       'Standard', 'Standard', 'Student', 'Student'], dtype='<U8')

In [12]:
age_means = np.array([23, 40, 52])
# I choose this stds becase students would be more concentrated around 23 years old while standard and premium
# segments would have more variaty
age_stds = np.array([3, 10, 10])

In [13]:
ages = np.clip(
    np.rint(
        rng.normal(
            loc = age_means[customer_segment_idx],
            scale = age_stds[customer_segment_idx]
        )
    ).astype(int),
    18,
    80
)

In [14]:
print(ages.min())
print(ages.max())

19
66


In [15]:
# I  create a different standard amount for each segment, customer
segment_base_amount = np.array([
    25.0, 60.0, 120.0
])

customer_base_amount = (
    segment_base_amount[customer_segment_idx]
    * rng.lognormal(mean = 0, sigma = 0.25, size = N_CUSTOMERS)
)

customer_base_amount[:5]

array([ 51.68370857,  18.65257428, 109.44580591,  65.34843226,
        92.43622741])

In [16]:
# I generate the prefered shopping hour of each customer
preferred_hours = np.mod(
        np.rint(
            rng.normal(
                loc = 15, # 15:00
                scale = 3,
                size = N_CUSTOMERS
            )
        ).astype(int),
        24
)

In [17]:
print(preferred_hours.min())
print(preferred_hours.max())

8
22


In [18]:
# Initial international prob / customer
base_international_probability = (
    np.array(
        [0.08, 0.12, 0.22] # Student, standard, premium
    )
)

In [19]:
international_probability = np.clip(
    base_international_probability[
        customer_segment_idx
    ]
    # I add an individual variation from a beta distribution
    + rng.beta( 2, 12, size=N_CUSTOMERS),
    0.02,
    0.65
)

In [20]:
# I also want some customers to appear in many transactions and vice versa, so I will use a gamma
# distribution to generate postive assymetric values which I'll turn into probabilities

# positive activity weight
activity_raw = rng.gamma(shape = 2.0, scale = 1.0, size = N_CUSTOMERS)

activity_probability = (activity_raw / activity_raw.sum())

In [21]:
activity_probability.sum() 

np.float64(1.0000000000000002)

In [22]:
# Preference vector / customer
category_preferences = rng.dirichlet(
    np.full(
        n_categories, 0.45 # < 1 for more concentrated preferences
    ),
    size = N_CUSTOMERS
)

In [23]:
category_preferences.sum(axis=1)[:5]

array([1., 1., 1., 1., 1.])

In [24]:
# I know generate how many days after 2019/01/01 each customer signed up
# I will cover 0-5 years
signup_days = rng.integers(0, 365*5, size=N_CUSTOMERS)

In [25]:
signup_dates = (pd.Timestamp("2019-01-01")
                + pd.to_timedelta(
                    signup_days, unit = "D"
                ))

In [26]:
# Customer table
customers = pd.DataFrame(
    {
        "customer_id": customer_ids,
        "age": ages,
        "country": countries[ customer_country_idx],
        "signup_date": signup_dates,
        "customer_segment": segments[customer_segment_idx],
        "typical_amount": np.round(customer_base_amount,2 ),
        "preferred_hour": (preferred_hours),
        "international_probability": (
            np.round(international_probability, 3)
        ),
        "activity_weight": np.round(activity_probability, 6)
    }
)

In [27]:
customers.head()

,customer_id,age,country,signup_date,customer_segment,typical_amount,preferred_hour,international_probability,activity_weight
0,C00001,26,Spain,2023-05-24,Student,51.68,10,0.321,0.026483
1,C00002,23,United Kingdom,2019-09-10,Student,18.65,19,0.403,0.009543
2,C00003,50,Ireland,2020-12-25,Premium,109.45,16,0.433,0.052133
3,C00004,30,Italy,2022-06-29,Standard,65.35,18,0.350,0.004717
4,C00005,23,Italy,2020-03-15,Standard,92.44,13,0.173,0.006435


In [28]:
customer_category_preferences = (pd.DataFrame(category_preferences, columns = categories))

In [29]:
customer_category_preferences.insert(0,"customer_id",customer_ids)

In [30]:
customer_category_preferences.head()

,customer_id,Groceries,Restaurants,Books,Electronics,Sports,Travel,Home,Entertainment,Fashion,Health
0,C00001,0.206261,0.120822,1.318096e-01,0.034190,0.076669,0.009607,0.026932,0.175351,0.017322,0.201037
1,C00002,0.127912,0.022040,2.636316e-02,0.381002,0.000479,0.139399,0.249938,0.001373,0.003777,0.047717
2,C00003,0.083236,0.100813,1.390525e-02,0.026400,0.078256,0.135766,0.163659,0.014220,0.146350,0.237395
3,C00004,0.004862,0.000783,1.567342e-08,0.224396,0.126763,0.005939,0.369622,0.079194,0.001587,0.186854
4,C00005,0.093042,0.092102,1.572832e-02,0.004447,0.001278,0.003067,0.135671,0.059135,0.018204,0.577326


In [31]:
# Now I will generate the accounts; I want 75% to have one account, and 25% to have two
account_counts = (1 + (rng.random(N_CUSTOMERS) < 0.25).astype(int))


In [32]:
account_customer_idx = np.repeat(np.arange(N_CUSTOMERS), account_counts)

In [33]:
N_ACCOUNTS = len(account_customer_idx)
N_ACCOUNTS

119

In [34]:
# ids for the accounts
accounts_ids = np.array([
    f"A{i:05d}" for i in range(1, N_ACCOUNTS + 1)
])

In [35]:
account_starts = np.cumsum(
    np.r_[0,account_counts[:-1]]
)

In [36]:
# position of the account
account_sequence = (
    np.arange(N_ACCOUNTS) - np.repeat(account_starts, account_counts)
)

In [37]:
# type of accounts -> 2nd account is for savings

account_types = np.where(
    account_sequence == 0,
    np.where(
        segments[customer_segment_idx[account_customer_idx]] # segment of each account's owner
    
    == "Student", 
    "Student",
    "Current"
    ),
    "Savings"
)

In [38]:
# opening date of the account

opening_dates = (
    signup_dates[account_customer_idx] + pd.to_timedelta(
        rng.integers(0, 365, size = N_ACCOUNTS),  unit = "D"
    )
)

In [39]:
accounts = pd.DataFrame(
    {
        "account_id": accounts_ids,

        "customer_id": customer_ids[account_customer_idx],

        "account_type": account_types,

        "currency": currencies[
            customer_country_idx[
                account_customer_idx
            ]
        ],
        "opening_date": opening_dates,

        "status": "Active",
    }
)

In [40]:
accounts.head()

,account_id,customer_id,account_type,currency,opening_date,status
0,A00001,C00001,Student,EUR,2023-09-25,Active
1,A00002,C00002,Student,GBP,2020-07-30,Active
2,A00003,C00002,Savings,GBP,2020-06-03,Active
3,A00004,C00003,Current,EUR,2021-07-11,Active
4,A00005,C00003,Savings,EUR,2021-07-10,Active


In [41]:
# Merchants catalogue
merchant_category_values = np.repeat(categories, n_countries)
merchant_country_values = np.tile(countries, n_categories)
merchant_currency_values = np.tile(currencies, n_categories)

In [42]:
N_MERCHANTS = len(merchant_category_values)
N_MERCHANTS

80

In [43]:
merchant_ids = np.array(
    [
        f"M{i:05d}" for i in range(1, N_MERCHANTS + 1)
    ]
)

In [44]:
merchant_names = np.array(
    [        f"{category} {country} Hub"
        for category, country in zip(
            merchant_category_values,
            merchant_country_values,
        )
    ]
)
merchant_names[:5]

array(['Groceries Spain Hub', 'Groceries France Hub',
       'Groceries Germany Hub', 'Groceries Italy Hub',
       'Groceries Portugal Hub'], dtype='<U32')

In [45]:
# Now I will define the online categories 
online_categories = np.array(
    [
        "Books",
        "Electronics",
        "Travel",
        "Home",
        "Entertainment",
        "Fashion",
    ]
)

In [46]:
merchant_online = np.isin(merchant_category_values, online_categories)

In [47]:
# Merchant dataframe

merchants = pd.DataFrame(
    {
        "merchant_id": merchant_ids,
        "merchant_name": merchant_names,
        "category": (
            merchant_category_values
        ),
        "country": (
            merchant_country_values
        ),
        "currency": (
            merchant_currency_values
        ),
        "online": merchant_online,
    }
)


In [48]:
merchants.head()

,merchant_id,merchant_name,category,country,currency,online
0,M00001,Groceries Spain Hub,Groceries,Spain,EUR,False
1,M00002,Groceries France Hub,Groceries,France,EUR,False
2,M00003,Groceries Germany Hub,Groceries,Germany,EUR,False
3,M00004,Groceries Italy Hub,Groceries,Italy,EUR,False
4,M00005,Groceries Portugal Hub,Groceries,Portugal,EUR,False


In [49]:
# Generation of transactions

# I will use activity_probability with rng.choice(), so for example a client with a 0.03 weight will have approximately
# 3 times more possibilities to be selected than one with a 0.01 weight

transaction_customer_idx = rng.choice(N_CUSTOMERS, size = N_TRANSACTIONS, p = activity_probability)

In [50]:
# What transactions will use the 2nd account
use_second_account = (
    (account_counts[transaction_customer_idx] == 2)
    & (rng.random(N_TRANSACTIONS) < 0.2)
)

In [51]:
transaction_account_position = (
    account_starts[ # where customer account starts
        transaction_customer_idx
    ]
    + use_second_account.astype(int) # +0 or 1 for the 1st or 2nd account
)

# I transform the positions in the transaction-account ids
transaction_account_ids = accounts_ids[transaction_account_position]

In [52]:
# Choose category according to preferences
selected_preferences = (
    category_preferences[
        transaction_customer_idx
    ]
)

In [53]:
category_cdf = np.cumsum(
    selected_preferences, axis=1,
)

In [54]:
# I will force the last limit of each row to 1 in order to avoid precision errors
category_cdf[:, -1] = 1.0

In [55]:
category_draws = rng.random( N_TRANSACTIONS)

In [56]:
transaction_category_idx = (
    category_draws[:, None] > category_cdf
).sum(axis=1)

In [57]:
# National or international country
customer_country_for_transaction = (customer_country_idx[transaction_customer_idx])

In [58]:
is_international = (rng.random(N_TRANSACTIONS) < international_probability[transaction_customer_idx])

In [59]:
# To guarantee the international transaction i need to generate a different country
international_offset = rng.integers(
    1, # min is 1 to avoid preserving the same country
    n_countries, size=N_TRANSACTIONS,
)

In [60]:
# Final country of the merchant
transaction_country_idx = np.where(
    is_international,  
    (customer_country_for_transaction + international_offset) % n_countries,
    customer_country_for_transaction,
)

In [61]:
# Transaction time
transaction_hours = np.mod(
    np.rint(
        rng.normal(
            loc=preferred_hours[ transaction_customer_idx
            ],
            scale=3,
            size=N_TRANSACTIONS,
        )
    ).astype(int),
    24,
)

In [62]:
# How much each category modifies the standard amount
category_multiplier = np.array(
    [
        0.7,      # Groceries
        0.6,         # Restaurants
        0.4,        # Books
        2.5,         # Electronics
        1.1,         # Sports
        3.0,         # Travel
        1.6,        # Home
        0.8,         # Entertainment
        1.2,         # Fashion
        0.5,         # Health
    ]
)

# Log mean / transaction
lognormal_mean = np.log(
    customer_base_amount[transaction_customer_idx] # std amount / customer
    * category_multiplier[transaction_category_idx] # adjusted per category
)

In [63]:
# Amount generation using a lognormal distribution
amounts = rng.lognormal( mean=lognormal_mean, sigma=0.55,)

In [64]:
# I assign the type of the transaction (purchase || refund)
transaction_types = np.where(
    rng.random(N_TRANSACTIONS) < 0.03, # approx 3% < 0.03
    "Refund",
    "Purchase",
)

In [65]:
# I compare each amount with the frequent amount of the customer
relative_amount = (amounts / customer_base_amount[ transaction_customer_idx])

In [66]:
# Prob of the transaction being declined

decline_probability = np.clip(
    0.02 # base prof 2%
    + 0.03 * (relative_amount > 3) # + 3% if it is 3 times larger than the frequent amount
    + 0.05 * (relative_amount > 7), # """" 7 times
    0,
    0.20, # max prob
)

In [67]:
statuses = np.where(
    rng.random(N_TRANSACTIONS) < decline_probability,
    "Declined",
    "Completed",
)

### Anomaly injection

In [68]:
# Candidates to the anomaly -- only COMPLETED PURCHASE transactions
anomaly_candidate = (transaction_types == "Purchase") & (statuses == "Completed")

In [69]:
anomaly_draw = rng.random(N_TRANSACTIONS)

In [70]:
"""0.000 - 0.005 high_amount
0.005 - 0.008 unusual_hour
0.008 - 0.010 unusual_category"""

high_amount_mask = (
    anomaly_candidate & (anomaly_draw < 0.005)
)

unusual_hour_mask = (
    anomaly_candidate & (anomaly_draw >= 0.005) & (anomaly_draw < 0.008)
)

unusual_category_mask = (
    anomaly_candidate & (anomaly_draw >= 0.008) & (anomaly_draw < 0.010)
)


In [71]:
# Anomalous amount

amounts[high_amount_mask] *= rng.uniform(
    8,
    20,
    size=high_amount_mask.sum(), # as many multipliers as number of anomalies 
)

In [72]:
# Anomalous hour
transaction_hours[unusual_hour_mask] = rng.integers(
    0,
    4,
    size=unusual_hour_mask.sum(),
)

In [73]:
# Anomalous category
# I first locate the least preferred category by the customer
least_preferred_category = (category_preferences[transaction_customer_idx]
    .argmin(axis=1)
)

In [74]:
transaction_category_idx[unusual_category_mask] = least_preferred_category[unusual_category_mask]

In [75]:
# Anomaly text label
anomaly_type = np.select(
    [
        high_amount_mask,
        unusual_hour_mask,
        unusual_category_mask,
    ],
    [
        "high_amount",
        "unusual_hour",
        "unusual_category",
    ],
    default="none"
)

In [76]:
is_injected_anomaly = (
    anomaly_type != "none"
)

### Merchant determination

Merchants are ordered like this:

category 0 * all countries

category 1 * all countries

Thus position is calculated with category * number of countries + country

In [77]:
merchant_position = (
    transaction_category_idx * n_countries
    + transaction_country_idx
)

In [78]:
# I then turn the position into merchant_id
transaction_merchant_ids = (
    merchant_ids[
        merchant_position
    ]
)

# Dates of the transactions

In [79]:
day_offsets = rng.integers(
    0,
    365,
    size=N_TRANSACTIONS,
)

minute_offsets = rng.integers(
    0,
    60,
    size=N_TRANSACTIONS,
)

In [80]:
# Timestamp
timestamps = (
    pd.Timestamp("2025-01-01")
    + pd.to_timedelta(  # days
        day_offsets,
        unit="D",
    )
    + pd.to_timedelta(  # hours
        transaction_hours,
        unit="h",
    )
    + pd.to_timedelta(  # minutes
        minute_offsets,
        unit="m",
    )
)

### Transaction table

In [81]:
transaction_ids = np.array(
    [f"T{i:07d}" for i in range(1, N_TRANSACTIONS + 1)
    ]
)

In [82]:
transactions = pd.DataFrame(
    {
        "transaction_id": (
            transaction_ids
        ),
        "account_id": (
            transaction_account_ids
        ),
        "merchant_id": (
            transaction_merchant_ids
        ),
        "timestamp": timestamps,
        "amount": np.round(
            amounts,
            2,
        ),
        "currency": currencies[
            transaction_country_idx
        ],
        "transaction_type": (
            transaction_types
        ),
        "status": statuses,

        "is_injected_anomaly": (
            is_injected_anomaly
        ),

        "anomaly_type": (
            anomaly_type
        ),
    }
)

In [83]:
transactions.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status,is_injected_anomaly,anomaly_type
0,T0000001,A00035,M00041,2025-02-27 16:45:00,257.70,EUR,Refund,Completed,False,none
1,T0000002,A00010,M00055,2025-02-25 12:45:00,38.90,GBP,Purchase,Completed,False,none
2,T0000003,A00055,M00062,2025-09-13 14:52:00,16.36,EUR,Purchase,Completed,False,none
3,T0000004,A00026,M00070,2025-07-06 08:05:00,26.03,EUR,Purchase,Completed,False,none
4,T0000005,A00110,M00060,2025-06-16 18:59:00,79.56,EUR,Purchase,Completed,False,none


In [84]:
print(
    "Customers:",
    len(customers),
)

print(
    "Accounts:",
    len(accounts),
)

print(
    "Merchants:",
    len(merchants),
)

print(
    "Transactions:",
    len(transactions),
)

Customers: 100
Accounts: 119
Merchants: 80
Transactions: 1000


In [85]:
# Rows per type of anomaly

transactions[
    "anomaly_type"
].value_counts()

anomaly_type
none                991
high_amount           7
unusual_category      1
unusual_hour          1
Name: count, dtype: int64

In [86]:
transactions.loc[
    transactions[
        "is_injected_anomaly"
    ]
].head(10)

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status,is_injected_anomaly,anomaly_type
117,T0000118,A00088,M00074,2025-06-03 13:46:00,313.42,EUR,Purchase,Completed,True,high_amount
277,T0000278,A00112,M00016,2025-08-27 18:42:00,702.78,USD,Purchase,Completed,True,high_amount
350,T0000351,A00048,M00039,2025-01-27 13:40:00,633.74,GBP,Purchase,Completed,True,high_amount
370,T0000371,A00112,M00016,2025-06-02 13:59:00,564.33,USD,Purchase,Completed,True,high_amount
496,T0000497,A00067,M00050,2025-08-23 13:17:00,13.75,EUR,Purchase,Completed,True,unusual_category
653,T0000654,A00058,M00055,2025-02-16 16:25:00,375.94,GBP,Purchase,Completed,True,high_amount
890,T0000891,A00058,M00055,2025-07-15 03:22:00,170.24,GBP,Purchase,Completed,True,unusual_hour
894,T0000895,A00033,M00052,2025-07-28 19:33:00,1175.90,EUR,Purchase,Completed,True,high_amount
925,T0000926,A00062,M00078,2025-04-20 16:41:00,1273.35,EUR,Purchase,Completed,True,high_amount


In [87]:
generation_validation = pd.Series(
    {
        "customer_ids_are_unique": customers["customer_id"].is_unique,
        "account_ids_are_unique": accounts["account_id"].is_unique,
        "merchant_ids_are_unique": merchants["merchant_id"].is_unique,
        "transaction_ids_are_unique": transactions["transaction_id"].is_unique,
        "account_foreign_keys_are_valid": transactions["account_id"].isin(accounts["account_id"]).all(),
        "merchant_foreign_keys_are_valid": transactions["merchant_id"].isin(merchants["merchant_id"]).all(),
        "amounts_are_positive": transactions["amount"].gt(0).all(),
        "timestamps_are_in_2025": transactions["timestamp"].dt.year.eq(2025).all(),
        "preference_rows_sum_to_one": np.isclose(category_preferences.sum(axis=1), 1.0).all(),
        "anomaly_labels_are_consistent": transactions["is_injected_anomaly"].eq(transactions["anomaly_type"].ne("none")).all(),
        "transaction_count_is_correct": len(transactions) == N_TRANSACTIONS,
        "transactions_have_no_missing_values": transactions.isna().sum().sum() == 0,
    },
    name="passed"
)

In [88]:
generation_validation.all()

np.True_

In [89]:
# I also want tocheck the memory usage of each column

transactions_memory_mb = (transactions.memory_usage(deep=True).sum()
    /1024**2
)

In [90]:
print(
    "Transactions memory:",
    round(
        transactions_memory_mb,
        3,
    ),
    "MB"
)

Transactions memory: 0.385 MB


In [91]:
# Save CSVs
customers_path = (
    synthetic_directory
    / "synthetic_customers.csv"
)

accounts_path = (
    synthetic_directory
    / "synthetic_accounts.csv"
)

preferences_path = (
    synthetic_directory
    / "synthetic_preferences.csv"
)

merchants_path = (
    synthetic_directory
    / "synthetic_merchants.csv"
)

transactions_path = (
    synthetic_directory
    / (
        f"synthetic_transactions_"
        f"{N_TRANSACTIONS}.csv"
    )
)

In [92]:
customers.to_csv(
    customers_path,
    index=False,
)

accounts.to_csv(
    accounts_path,
    index=False,
)

merchants.to_csv(
    merchants_path,
    index=False,
)

customer_category_preferences.to_csv(
    preferences_path,
    index=False,
)

transactions.to_csv(
    transactions_path,
    index=False,
)

In [93]:
# Main metadata summary
generation_summary = pd.Series(
    {
        "seed": SEED,
        "dataset_tier": DATASET_TIER,
        "customers": len(customers),
        "accounts": len(accounts),
        "merchants": len(merchants),
        "transactions": len(
            transactions
        ),
        "injected_anomalies": int(
            transactions[
                "is_injected_anomaly"
            ].sum()
        ),
        "transaction_memory_mb": round(
            transactions_memory_mb,
            3,
        ),
    },
    name="value"
)

In [94]:
generation_summary

seed                              42
dataset_tier             development
customers                        100
accounts                         119
merchants                         80
transactions                    1000
injected_anomalies                 9
transaction_memory_mb          0.385
Name: value, dtype: object

In [95]:
generation_summary.to_csv(
    synthetic_directory
    / "generation_summary.csv",
    header=True,
)